# B4. Kill-and-restart test for the model runner

Starts a 50-row run with Qwen3-VL, kills it after at least 20 rows are written, starts it again, and checks the file ends with exactly 50 rows and no repeats.

**Before you press Run all:** Accelerator **GPU T4 x1**, Internet **on**, and the `GITHUB_TOKEN` secret ticked for this notebook (the same one as B3).

In [ ]:
!pip install -q -U 'transformers>=4.57' accelerate bitsandbytes pillow num2words pyyaml
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, turn on the GPU in settings')

## 1. Get the code

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 2. A 50-row manifest
All 50 tuning photos, clean, resized to a 512 px short side.

In [ ]:
import io, urllib.request
import pandas as pd
from PIL import Image

atk = pd.read_csv('data/attack_set.csv')
pick = atk[atk['split'] == 'tune']
os.makedirs('/tmp/b4/images', exist_ok=True)
rows = []
for r in pick.itertuples():
    with urllib.request.urlopen('http://images.cocodataset.org/val2017/' + r.file_name, timeout=30) as f:
        im = Image.open(io.BytesIO(f.read())).convert('RGB')
    s = 512 / min(im.size)
    im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS).save(f'/tmp/b4/images/{r.image_id}_clean_none.png')
    rows.append({'image_id': r.image_id, 'cell': 'clean', 'setting': 'none', 'defence': 'none', 'path': f'images/{r.image_id}_clean_none.png'})
pd.DataFrame(rows).to_csv('/tmp/b4/manifest.csv', index=False)
print(len(rows), 'rows in the manifest')

## 3. Start the run and kill it after 20 rows
The run starts in the background. This cell waits until the output file has at least 20 rows, then kills the process without warning, as a crash or a Kaggle timeout would.

In [ ]:
import time
OUT = '/kaggle/working/runs/b4_resume_qwen3-vl-4b.jsonl'
os.makedirs('/kaggle/working/runs', exist_ok=True)
if os.path.exists(OUT):
    os.remove(OUT)
CMD = ['python', 'src/run_model.py', '--model', 'qwen3-vl-4b', '--manifest', '/tmp/b4/manifest.csv',
       '--images-root', '/tmp/b4', '--out', OUT]

def rows_written():
    return sum(1 for _ in open(OUT, encoding='utf-8')) if os.path.exists(OUT) else 0

first = subprocess.Popen(CMD, stdout=open('/tmp/b4/first.log', 'w'), stderr=subprocess.STDOUT)
start = time.time()
while rows_written() < 20 and first.poll() is None and time.time() - start < 900:
    time.sleep(0.1)
first.kill()
first.wait()
stopped_at = rows_written()
print('killed after', round(time.time() - start), 's with', stopped_at, 'rows written')
print('OK, stopped part-way' if 20 <= stopped_at < 50 else 'PROBLEM: the run was not stopped part-way')
if stopped_at < 20:
    print(open('/tmp/b4/first.log', encoding='utf-8').read()[-2000:])

## 4. Start it again

In [ ]:
!python src/run_model.py --model qwen3-vl-4b --manifest /tmp/b4/manifest.csv --images-root /tmp/b4 --out {OUT} 2>&1 | grep -v -E 'Loading weights|Downloading|Reconstruct|Fetching|it/s'

## 5. Check the result
Exactly 50 rows, no repeats, every photo there once and one config hash.

In [ ]:
df = pd.read_json(OUT, lines=True)
key = ['image_id', 'cell', 'setting', 'defence']
checks = {
    'exactly 50 rows': len(df) == 50,
    'no repeated rows': not df.duplicated(key).any(),
    'every photo once': sorted(df['image_id']) == sorted(r['image_id'] for r in rows),
    'one config hash': df['config_hash'].nunique() == 1,
}
for k, v in checks.items():
    print(f'{k:28s} {v}')
print('rows from the first run:', stopped_at, '| rows from the restart:', len(df) - stopped_at)
print('OK' if all(checks.values()) else 'PROBLEM')